In [31]:
# ==========================================
# 1. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS
# ==========================================
!pip install shap xgboost imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc
)
import shap

# Configurações visuais
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [32]:
# ==========================================
# 1. INSTALAÇÃO E IMPORTAÇÃO DE BIBLIOTECAS
# ==========================================
!pip install shap xgboost imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc
)
import shap

# Configurações visuais
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

In [33]:
import pandas as pd

# Usando um arquivo de exemplo que já existe no seu Google Colab
caminho_do_arquivo = '/content/sample_data/california_housing_test.csv'

# Carregando o dataset
df = pd.read_csv(caminho_do_arquivo)

# Exibindo as primeiras linhas para verificar se foi carregado corretamente
display(df.head())

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.05,37.37,27.0,3885.0,661.0,1537.0,606.0,6.6085,344700.0
1,-118.30,34.26,43.0,1510.0,310.0,809.0,277.0,3.5990,176500.0
2,-117.81,33.78,27.0,3589.0,507.0,1484.0,495.0,5.7934,270500.0
3,-118.36,33.82,28.0,67.0,15.0,49.0,11.0,6.1359,330000.0
4,-119.67,36.33,19.0,1241.0,244.0,850.0,237.0,2.9375,81700.0


In [34]:
# Exibir a lista de colunas disponíveis no DataFrame atual
print("Colunas no DataFrame:")
print(df.columns.tolist())

# Também exibiremos informações gerais das colunas, incluindo tipos de dados
df.info()

Colunas no DataFrame:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           3000 non-null   float64
 1   latitude            3000 non-null   float64
 2   housing_median_age  3000 non-null   float64
 3   total_rooms         3000 non-null   float64
 4   total_bedrooms      3000 non-null   float64
 5   population          3000 non-null   float64
 6   households          3000 non-null   float64
 7   median_income       3000 non-null   float64
 8   median_house_value  3000 non-null   float64
dtypes: float64(9)
memory usage: 211.1 KB


In [ ]:
from google.colab import files
import io
import pandas as pd

print("Por favor, execute esta célula e clique no botão abaixo para selecionar o arquivo 'creditcard.csv':")
uploaded = files.upload()

if uploaded:
    nome_arquivo = list(uploaded.keys())[0]
    df = pd.read_csv(io.BytesIO(uploaded[nome_arquivo]))
    print(f"\nSucesso! O arquivo '{nome_arquivo}' foi carregado no DataFrame 'df'.")
    display(df.head())
else:
    print("\nNenhum arquivo foi carregado. Por favor, tente novamente.")

Por favor, execute esta célula e clique no botão abaixo para selecionar o arquivo 'creditcard.csv':


In [ ]:
import pandas as pd
import os

caminho_local = 'creditcard.csv'

# Verifica se o arquivo está no diretório atual
if os.path.exists(caminho_local):
    print(f"Carregando arquivo local: {caminho_local}")
    df = pd.read_csv(caminho_local)
else:
    print("Arquivo local não encontrado. Carregando dataset a partir de uma URL pública alternativa...")
    # Link público direto, estável e funcional do dataset creditcard.csv
    url_publica = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
    df = pd.read_csv(url_publica)

print(f"\nSucesso! O DataFrame 'df' foi populado com {df.shape[0]} linhas e {df.shape[1]} colunas.")
display(df.head())

In [ ]:
# ==========================================
# 3. ENGENHARIA DE RECURSOS E PRÉ-PROCESSAMENTO
# ==========================================
# Criação do log do valor da transação (reduz assimetria)
df['Amount_log'] = np.log1p(df['Amount'])

# Padronização das variáveis escalares
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount_log']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

# Separação de X e y
features = [col for col in df.columns if col not in ['Class', 'Amount', 'Time', 'Amount_log']]
X = df[features]
y = df['Class']

# Divisão em Treino e Teste garantindo a proporção da classe minoritária (stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Tamanho do treino: {X_train.shape[0]} | Tamanho do teste: {X_test.shape[0]}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Definindo o tamanho da figura
plt.figure(figsize=(10, 6))

# Criando o gráfico de distribuição usando a coluna 'Amount' (que existe no dataset de cartões)
sns.histplot(df['Amount'], kde=True, color='blue', bins=50)

# Ajustando escala para logarítmica devido à grande variação dos valores de transação
plt.yscale('log')

# Customizações do gráfico
plt.title('Distribuição dos Valores das Transações (Escala Logarítmica no Eixo Y)', fontsize=14)
plt.xlabel('Valor da Transação ($)', fontsize=12)
plt.ylabel('Frequência (Log)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)

# Exibindo o gráfico
plt.show()

In [ ]:
# ==========================================
# 4. TREINAMENTO DOS MODELOS
# ==========================================
# Proporção para ajuste do peso das classes (class_weight / scale_pos_weight)
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

# 1. Regressão Logística (Baseline)
model_lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model_lr.fit(X_train, y_train)

# 2. Random Forest
model_rf = RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

# 3. XGBoost
model_xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
model_xgb.fit(X_train, y_train)

In [ ]:
# ==========================================
# 4. TREINAMENTO DOS MODELOS (EXECUÇÃO)
# ==========================================

# Proporção para ajuste do peso das classes (class_weight / scale_pos_weight)
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
print(f"Fator de escala de peso positivo para XGBoost: {scale_pos_weight:.2f}\n")

# 1. Regressão Logística (Baseline)
print("Treinando Regressão Logística...")
model_lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model_lr.fit(X_train, y_train)
print("Regressão Logística treinada com sucesso!")

# 2. Random Forest
print("\nTreinando Random Forest (isso pode levar um minuto)...")
model_rf = RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)
print("Random Forest treinado com sucesso!")

# 3. XGBoost
print("\nTreinando XGBoost...")
model_xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss')
model_xgb.fit(X_train, y_train)
print("XGBoost treinado com sucesso!")

In [ ]:
# ==========================================
# 5. AVALIAÇÃO DOS MODELOS
# ==========================================
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, auc

modelos = {
    'Regressão Logística': model_lr,
    'Random Forest': model_rf,
    'XGBoost': model_xgb
}

for nome, modelo in modelos.items():
    print(f"=== {nome} ===")
    # Predições
    y_pred = modelo.predict(X_test)

    # Probabilidades para cálculo de curvas (caso o modelo possua predict_proba)
    if hasattr(modelo, "predict_proba"):
        y_probs = modelo.predict_proba(X_test)[:, 1]
        # Cálculo do AUPRC
        precision, recall, _ = precision_recall_curve(y_test, y_probs)
        auprc = auc(recall, precision)
        roc_auc = roc_auc_score(y_test, y_probs)
        print(f"ROC AUC: {roc_auc:.4f}")
        print(f"AUPRC (Área sob Curva PR): {auprc:.4f}")

    # Relatório de Classificação
    print("\nRelatório de Classificação:")
    print(classification_report(y_test, y_pred))

    print("Matriz de Confusão:")
    print(confusion_matrix(y_test, y_pred))
    print("-" * 50 + "\n")

In [ ]:
# ==========================================
# 5. AVALIAÇÃO COMPARATIVA E AJUSTE DE LIMIAR
# ==========================================
models = {
    'Logistic Regression': model_lr,
    'Random Forest': model_rf,
    'XGBoost': model_xgb
}

results = []

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for name, model in models.items():
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    # Métricas
    report = classification_report(y_test, y_pred, output_dict=True)
    rec = report['1']['recall']
    prec = report['1']['precision']
    f1 = report['1']['f1-score']
    roc_auc = roc_auc_score(y_test, y_pred_prob)

    precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_pred_prob)
    pr_auc = auc(recall_curve, precision_curve)

    results.append({
        'Modelo': name,
        'Recall (Fraude)': rec,
        'Precision (Fraude)': prec,
        'F1-Score (Fraude)': f1,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc
    })

    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    ax1.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})')

    # Curva Precision-Recall
    ax2.plot(recall_curve, precision_curve, label=f'{name} (PR-AUC = {pr_auc:.3f})')

ax1.set_title('Curva ROC')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend()

ax2.set_title('Curva Precision-Recall')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.legend()

plt.show()

# Tabela Comparativa
df_results = pd.DataFrame(results)
display(df_results)

In [ ]:
# ==========================================
# 6. EXPLICABILIDADE DO MODELO (SHAP)
# ==========================================
# Explicando o modelo XGBoost com SHAP
explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer(X_test.iloc[:500]) # Amostra para agilizar o cálculo

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.iloc[:500], plot_type="bar")

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test.iloc[:500])